In [1]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
import requests
import time
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("data")
if not (DATA_DIR / "tmdb_5000_movies.csv").exists():
    DATA_DIR = Path("data") / "data"

In [2]:
df = pd.read_csv(DATA_DIR / "tmdb_5000_movies.csv")

df.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [3]:
df = df[['title', 'budget', 'runtime']].drop_duplicates()

df = df[(df['budget'] > 0) & (df['runtime'] > 0)]

df.head()

,title,budget,runtime
0,Avatar,237000000,162.0
1,Pirates of the Caribbean: At World's End,300000000,169.0
2,Spectre,245000000,148.0
3,The Dark Knight Rises,250000000,165.0
4,John Carter,260000000,132.0


In [4]:
API_KEY = os.getenv("OMDB_API_KEY")

def get_imdb_rating(title):
    if not API_KEY:
        raise RuntimeError("Set OMDB_API_KEY before refreshing ratings.")

    response = requests.get(
        "https://www.omdbapi.com/",
        params={"apikey": API_KEY, "t": title},
        timeout=15,
    )
    response.raise_for_status()
    return response.json().get("imdbRating")

In [5]:
def safe_get_rating(title):
    time.sleep(1)
    return get_imdb_rating(title)

cleaned_path = DATA_DIR / "cleaned_movies.csv"
refresh_ratings = os.getenv("REFRESH_OMDB_RATINGS") == "1"

if cleaned_path.exists() and not refresh_ratings:
    df_subset = pd.read_csv(cleaned_path)
else:
    # OMDb's free tier is rate limited, so fetch one title per second.
    df_subset = df.head(500).copy()
    df_subset['imdb_rating'] = df_subset['title'].apply(safe_get_rating)

In [6]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
# Remove rows with 'N/A' or null ratings
df_subset = df_subset[df_subset['imdb_rating'] != 'N/A']
df_subset = df_subset.dropna(subset=['imdb_rating'])

# Convert to float
df_subset['imdb_rating'] = df_subset['imdb_rating'].astype(float)

# Save to CSV
df_subset.to_csv(cleaned_path, index=False)

df_subset.head()

,title,budget,runtime,imdb_rating
0,Avatar,237000000,162.0,7.9
1,Pirates of the Caribbean: At World's End,300000000,169.0,7.1
2,Spectre,245000000,148.0,6.8
3,The Dark Knight Rises,250000000,165.0,8.4
4,John Carter,260000000,132.0,6.6


In [8]:
df = pd.read_csv(cleaned_path)

# Features and target
X = df[['budget', 'runtime']]
y = df['imdb_rating']

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape

((392, 2), (99, 2))

In [9]:
model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

y_pred[:5]

array([6.50392189, 6.69044362, 6.54820479, 6.76667671, 6.15175992])

In [10]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: {mae:.3f}")
print(f"Mean Squared Error: {mse:.3f}")
print(f"R2 Score: {r2:.3f}")

Mean Absolute Error: 0.676
Mean Squared Error: 0.705
R2 Score: 0.213


In [11]:
df_subset.info()
df_subset.describe()

<class 'pandas.DataFrame'>
RangeIndex: 491 entries, 0 to 490
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        491 non-null    str    
 1   budget       491 non-null    int64  
 2   runtime      491 non-null    float64
 3   imdb_rating  491 non-null    float64
dtypes: float64(2), int64(1), str(1)
memory usage: 15.5 KB


,budget,runtime,imdb_rating
count,4.910000e+02,491.000000,491.000000
mean,1.234812e+08,118.427699,6.628106
std,4.871546e+07,22.784177,0.983986
min,8.000000e+06,74.000000,2.300000
25%,8.750000e+07,101.000000,6.100000
50%,1.100000e+08,115.000000,6.600000
75%,1.500000e+08,132.000000,7.300000
max,3.800000e+08,201.000000,9.000000


In [12]:
plt.figure(figsize=(8, 5))
sns.heatmap(df_subset.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

C:\Users\tripa\AppData\Local\Temp\ipykernel_18812\3758596536.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
df_subset['budget_millions'] = df_subset['budget'] / 1_000_000

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_subset, x='budget_millions', y='imdb_rating')
plt.title('Budget (Millions) vs IMDb Rating')
plt.xlabel('Budget (in Millions USD)')
plt.ylabel('IMDb Rating')
plt.show()

C:\Users\tripa\AppData\Local\Temp\ipykernel_18812\2980593060.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
plt.figure(figsize=(8, 5))
sns.histplot(df_subset['imdb_rating'], bins=20, kde=True)
plt.title('Distribution of IMDb Ratings')
plt.xlabel('IMDb Rating')
plt.ylabel('Frequency')
plt.show()

C:\Users\tripa\AppData\Local\Temp\ipykernel_18812\1253933019.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=df_subset['imdb_rating'])
plt.title("Boxplot of IMDb Ratings")
plt.xlabel("IMDb Rating")
plt.show()

C:\Users\tripa\AppData\Local\Temp\ipykernel_18812\4081172735.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
tmdb = pd.read_csv(DATA_DIR / "tmdb_5000_movies.csv", quotechar='"')
genre_df = tmdb[['title', 'genres']].drop_duplicates()
genre_df.head()

,title,genres
0,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam..."
1,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""..."
2,Spectre,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam..."
3,The Dark Knight Rises,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam..."
4,John Carter,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam..."


In [17]:
import ast

def extract_genre_names(genre_str):
    try:
        genre_list = ast.literal_eval(genre_str)
        return [genre['name'] for genre in genre_list]
    except:
        return []

genre_df['genre_list'] = genre_df['genres'].apply(extract_genre_names)
genre_df[['title', 'genre_list']].head()

,title,genre_list
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]"
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]"
2,Spectre,"[Action, Adventure, Crime]"
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]"
4,John Carter,"[Action, Adventure, Science Fiction]"


In [18]:
# Create genre dummies
genre_dummies = genre_df['genre_list'].str.join('|').str.get_dummies(sep='|')

# Merge genre dummies with titles
genre_encoded = pd.concat([genre_df['title'], genre_dummies], axis=1)

# Show a few rows
genre_encoded.head()

,title,Action,Adventure,Animation,Comedy,Crime,Documentary,Drama,Family,Fantasy,...,History,Horror,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western
0,Avatar,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,1,0,0,0,0
1,Pirates of the Caribbean: At World's End,1,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,Spectre,1,1,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,The Dark Knight Rises,1,0,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,1,0,0
4,John Carter,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [19]:
df_merged = pd.merge(df_subset, genre_encoded, on='title', how='inner')

# Check updated shape and sample
print("Shape after merging:", df_merged.shape)
df_merged.head()

Shape after merging: (491, 25)


,title,budget,runtime,imdb_rating,budget_millions,Action,Adventure,Animation,Comedy,Crime,...,History,Horror,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western
0,Avatar,237000000,162.0,7.9,237.0,1,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,Pirates of the Caribbean: At World's End,300000000,169.0,7.1,300.0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Spectre,245000000,148.0,6.8,245.0,1,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,The Dark Knight Rises,250000000,165.0,8.4,250.0,1,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
4,John Carter,260000000,132.0,6.6,260.0,1,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [20]:
# Exclude the plotting-only budget_millions duplicate from model features.
X = df_merged.drop(columns=['title', 'imdb_rating', 'budget_millions'])

# Target
y = df_merged['imdb_rating']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model
model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Evaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.3f}")
print(f"MSE: {mse:.3f}")
print(f"R2 Score: {r2:.3f}")

MAE: 0.653
MSE: 0.645
R2 Score: 0.280


In [21]:
from sklearn.ensemble import RandomForestRegressor

# Model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

# Evaluation
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest MAE: {mae_rf:.3f}")
print(f"Random Forest MSE: {mse_rf:.3f}")
print(f"Random Forest R2 Score: {r2_rf:.3f}")

Random Forest MAE: 0.690
Random Forest MSE: 0.723
Random Forest R2 Score: 0.193


In [22]:
from sklearn.model_selection import cross_val_score

# Evaluate R2 using 5-fold cross-validation
cv_r2 = cross_val_score(model, X, y, cv=5, scoring='r2')
cv_mae = cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error')
cv_mse = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')

print(f"Cross-Validated R2: {cv_r2.mean():.3f}")
print(f"Cross-Validated MAE: {-cv_mae.mean():.3f}")
print(f"Cross-Validated MSE: {-cv_mse.mean():.3f}")

Cross-Validated R2: 0.179
Cross-Validated MAE: 0.660
Cross-Validated MSE: 0.744


In [23]:
from sklearn.model_selection import GridSearchCV

# Define hyperparameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
}

rf_grid = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(estimator=rf_grid, param_grid=param_grid,
                           cv=5, scoring='r2', n_jobs=1)

# Fit grid search
grid_search.fit(X_train, y_train)

# Best model
best_rf = grid_search.best_estimator_
y_pred_best_rf = best_rf.predict(X_test)

# Evaluation
mae_best = mean_absolute_error(y_test, y_pred_best_rf)
mse_best = mean_squared_error(y_test, y_pred_best_rf)
r2_best = r2_score(y_test, y_pred_best_rf)

print("Best Parameters:", grid_search.best_params_)
print(f"Tuned RF MAE: {mae_best:.3f}")
print(f"Tuned RF MSE: {mse_best:.3f}")
print(f"Tuned RF R2 Score: {r2_best:.3f}")

Best Parameters: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}
Tuned RF MAE: 0.673
Tuned RF MSE: 0.700
Tuned RF R2 Score: 0.219


In [24]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x=y_test, y=y_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--')
plt.xlabel("True IMDb Rating")
plt.ylabel("Predicted IMDb Rating")
plt.title("Linear Regression: True vs. Predicted Ratings")
plt.show()

C:\Users\tripa\AppData\Local\Temp\ipykernel_18812\3666676361.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [25]:
residuals = y_test - y_pred

plt.figure(figsize=(8, 5))
sns.histplot(residuals, bins=20, kde=True)
plt.title("Distribution of Residuals (Linear Regression)")
plt.xlabel("Residual (True - Predicted)")
plt.ylabel("Frequency")
plt.axvline(0, color='red', linestyle='--')
plt.show()

C:\Users\tripa\AppData\Local\Temp\ipykernel_18812\640753100.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [26]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "Tuned Random Forest"],
    "MAE": [mae, mae_rf, mae_best],
    "MSE": [mse, mse_rf, mse_best],
    "R2 Score": [r2, r2_rf, r2_best]
})

results

,Model,MAE,MSE,R2 Score
0,Linear Regression,0.653305,0.645261,0.280046
1,Random Forest,0.690222,0.723285,0.192990
2,Tuned Random Forest,0.672623,0.699618,0.219397
